<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-II/blob/main/Clase%207/Ejemplo_2_lluvia_en_cba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# datos espaciales
### Coderhouse - Data Science
Profe Jorge Ruiz

In [ ]:
#pip install geopandas

In [ ]:
import geopandas as gpd


# Ruta del archivo TopoJSON
ruta_archivo = 'departamentos-cordoba.topojson'

# Leer el archivo TopoJSON como un DataFrame de GeoPandas
gdf = gpd.read_file(ruta_archivo, driver='TopoJSON')

# Visualizar el mapa
gdf.plot()

# otros topojson de Argentina pueden encontrarlos en https://github.com/mgaitan/departamentos_argentina

gdf

In [ ]:
import pandas as pd

# Ruta del archivo CSV de precipitaciones
ruta_precipitaciones = 'Precipitaciones Córdoba.csv'

# Leer el archivo CSV
df_precipitaciones = pd.read_csv(ruta_precipitaciones, encoding='latin-1', sep=";")
df_precipitaciones

# precipitaciones para el mes de abril 2021

In [ ]:
# hay nombres de departamentos con errores

df_precipitaciones.replace("PTE ROQUE SAENZ PENIA","PRESIDENTE ROQUE SAENZ PEÑA", inplace=True)

df_precipitaciones.replace("GRAL SAN MARTIN","GENERAL SAN MARTIN", inplace=True)
df_precipitaciones

In [ ]:
# Unir los datos de lluvia con el DataFrame de GeoPandas

gdf_merge = gdf.merge(df_precipitaciones, on='departamento', how='inner')
gdf_merge

In [ ]:
len(df_precipitaciones)== len(gdf_merge)

#verificar que el total de filas sea igual, sino falto modificar algún nombre

In [ ]:
# Visualizar el mapa coroplético
import matplotlib.pyplot as plt

# Crear el gráfico de los departamentos
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column='lluvia', ax=ax, color='white', edgecolor='black')

# Agregar la capa de lluvia
gdf_merge.plot(ax=ax, column='lluvia', cmap='Blues', linewidth=0.8, edgecolor='0.8', legend=True)

# Mostrar el gráfico
plt.show()




In [ ]:
# Guardar el gráfico en un archivo de imagen si lo necesitamos para un informe, presentación, etc.
fig.savefig('mapa_lluvia_cordoba.png')

In [ ]:
# Crear el mapa choropleth con plotly express
import geopandas as gpd
import plotly.express as px

fig = px.choropleth_mapbox(
    gdf_merge,
    geojson=gdf_merge.geometry.__geo_interface__,
    locations=gdf_merge.index,  # Utiliza el índice del GeoDataFrame como ubicaciones
    color='lluvia',
    color_continuous_scale='YlGnBu',
    hover_name=gdf_merge.index,
    mapbox_style="carto-positron",
    center={"lat": -32, "lon": -64},
    zoom=5,
    opacity=0.7,
    labels={'lluvia': 'Lluvia'},
    title='Mapa de Precipitaciones por departamentos en Córdoba'
)

fig.show()

In [ ]:
gdf_merge.set_index('departamento', inplace=True)

In [ ]:
import plotly.graph_objects as go


gdf_merge.lluvia= gdf_merge.lluvia.astype(int)
gdf_merge= gdf_merge.sort_values(by="lluvia") # vamos a darle un orden a los registros de lluvia desc

# Create a bar plot with the sorted departments on the y-axis
fig = go.Figure(data=[go.Bar(
    y=gdf_merge.index,
    x=gdf_merge['lluvia'],
    orientation='h',
    text=gdf_merge['lluvia'],
     textposition='outside'

)])

# Update the layout of the plot
fig.update_layout(
    title='Acumulados de precipitaciones por departamentos en la provincia de Córdoba',
    xaxis_title='Lluvia (mm)',
    yaxis_title='Departamentos',
    yaxis_categoryarray=gdf_merge.index,
    #yaxis_tickangle= -45
)

# Show the plot
fig.show()



revisar más info en https://plotly.com/python/creating-and-updating-figures/

In [ ]:
fig.write_html("file.html")